# Cross-Product Review: USD Funding Conditions (Seiche) + Exit Liquidity (Undertow)

Build one source-separated financial evidence packet combining:
1. **Seiche** (`seiche.info`) for system funding, SOFR-IORB, and money-market context.
2. **Undertow** (`liquilens-undertow.com`) for market depth, corporate credit spreads (HY/IG OAS), and stated-size exit context.

Each product retains its own retrieval clock, observation dates, SHA-256 digest, and evidence boundaries. Readings are never averaged into a universal market score.

- Seiche funding observatory: https://seiche.info/
- Undertow liquidity observatory: https://liquilens-undertow.com/
- Connected workflow guide: https://liquilens.in/start/

**Boundary:** Public research context only, not an executable quote, trade recommendation, or investment advice.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from urllib.request import Request, urlopen

ENDPOINTS = {
    "undertow_summary": "https://api.seiche.info/undertow/x402/summary",
    "liquilens_markets": "https://api.liquilens.in/api/evidence/markets",
    "seiche_capital_markets": "https://api.seiche.info/api/v2/world-markets?section=capital_markets",
}

packet = {}
for name, url in ENDPOINTS.items():
    retrieved = datetime.now(timezone.utc).isoformat()
    try:
        req = Request(url, headers={"Accept": "application/json", "User-Agent": "FinancialEvidence-Colab/1.0"})
        with urlopen(req, timeout=20) as resp:
            raw = resp.read()
        packet[name] = {
            "status": "ok",
            "url": url,
            "retrieved_utc": retrieved,
            "bytes": len(raw),
            "sha256": hashlib.sha256(raw).hexdigest(),
            "payload": json.loads(raw.decode("utf-8")),
        }
    except Exception as exc:
        packet[name] = {
            "status": "unavailable",
            "url": url,
            "retrieved_utc": retrieved,
            "error": f"{type(exc).__name__}: {exc}",
        }

for name, rec in packet.items():
    if rec["status"] == "ok":
        p = rec["payload"]
        print(f"[OK] {name}: {rec['bytes']} bytes, sha256={rec['sha256'][:12]}, asof={p.get('asof') or p.get('as_of')}")
    else:
        print(f"[UNAVAILABLE] {name}: {rec['error']}")

In [ ]:
# Inspect Undertow segment liquidity states without flattening them into one score
u = packet.get("undertow_summary", {})
if u.get("status") == "ok":
    data = u["payload"]
    print("Undertow asof:", data.get("asof"))
    print("Funding regime:", data.get("funding_regime"))
    print("Segments:", json.dumps(data.get("segments"), indent=2)[:1500])
else:
    print("Undertow summary unavailable:", u.get("error"))